In [17]:
import os, glob

print("Working directory:", os.getcwd())
print("CSV files I can see from here:")
for f in glob.glob("**/*.csv", recursive=True):
    print("   ", f)

Working directory: d:\SLIIT\Final Research\congintive_train_system\Notebook
CSV files I can see from here:


In [3]:
import pandas as pd

In [9]:
PATH = r"D:\SLIIT\Final Research\congintive_train_system\data\investigator_ftldlbd_nacc74.csv"

ID, VISIT, VER = "NACCID", "NACCVNUM", "FORMVER"

# --- 5 domains -> variables, with valid score ranges (out-of-range -> missing) ---
NEURO = {
    "Memory":       ["CRAFTVRS", "CRAFTURS", "CRAFTDVR", "CRAFTDRE", "UDSBENTD"],
    "Attention/WM": ["DIGFORCT", "DIGFORSL", "DIGBACCT", "DIGBACLS", "TRAILA"],
    "Executive":    ["TRAILB", "UDSVERFC", "UDSVERLC", "UDSVERTN"],
    "Language":     ["MINTTOTS", "ANIMALS", "VEG"],
    "Visuospatial": ["UDSBENTC"],
}
RANGES = {
    "CRAFTVRS": (0, 44), "CRAFTURS": (0, 25), "CRAFTDVR": (0, 44), "CRAFTDRE": (0, 25),
    "UDSBENTD": (0, 17), "UDSBENTC": (0, 17),
    "DIGFORCT": (0, 16), "DIGFORSL": (0, 9), "DIGBACCT": (0, 14), "DIGBACLS": (0, 8),
    "TRAILA": (0, 150), "TRAILB": (0, 300),
    "UDSVERFC": (0, 40), "UDSVERLC": (0, 40), "UDSVERTN": (0, 80),
    "MINTTOTS": (0, 32), "ANIMALS": (0, 77), "VEG": (0, 77),
}
DX = ["NACCUDSD", "NACCALZD", "NACCETPR", "NACCLBDM", "NACCFTD"]  # diagnosis / module flags
GEN = ["NACCNE4S", "NACCAPOE"]                                   # genetics
NP  = ["NACCAUTP", "NACCLEWY", "NACCBRAA"]                       # autopsy ground truth

WANT = [ID, VISIT, VER] + sum(NEURO.values(), []) + DX + GEN + NP


In [10]:
print(os.path.exists(PATH))   # must print True

True


In [11]:
header = pd.read_csv(PATH, nrows=0)
present = [c for c in WANT if c in header.columns]
missing = [c for c in WANT if c not in header.columns]
df = pd.read_csv(PATH, usecols=present, low_memory=False)
for c in df.columns:
    if c not in (ID, VER):
        df[c] = pd.to_numeric(df[c], errors="coerce")

print(f"\nFile: {df.shape[0]:,} visit-rows x {len(present)} of {len(WANT)} wanted columns")
print(f"Unique participants: {df[ID].nunique():,}")
if missing:
    print(f"NOT in file (note for): {missing}")


File: 217,598 visit-rows x 31 of 31 wanted columns
Unique participants: 57,038


In [7]:
print("\nFORMVER distribution (all rows):")
print(df[VER].astype(str).str.strip().value_counts(dropna=False).to_string())
v = df[VER].astype(str).str.strip()
c = df[v.str.startswith("3") | v.str.startswith("4")].copy()
ids = c[ID].nunique()
print(f"\nKept UDS v3+v4: {c.shape[0]:,} visits across {ids:,} participants")



FORMVER distribution (all rows):
FORMVER
3.0    94802
2.0    80120
1.0    24005
3.2    12074
4.0     6597

Kept UDS v3+v4: 113,473 visits across 34,273 participants


In [13]:
v = df[VER].astype(str).str.strip()
uds = df[v.str.startswith("3") | v.str.startswith("4")].copy()
ids = uds[ID].nunique()
print(f"Kept UDS v3+v4: {uds.shape[0]:,} visits across {ids:,} participants")

Kept UDS v3+v4: 113,473 visits across 34,273 participants


In [14]:
print("===== COGNITIVE DOMAIN COVERAGE (v3+v4) =====")
for domain, cols in NEURO.items():
    print(f"\n{domain}")
    best_people = 0
    for col in cols:
        if col not in uds.columns:
            print(f"   {col:<10}  not in file")
            continue
        lo, hi = RANGES[col]
        valid = uds[col].between(lo, hi)
        ppl = uds.loc[valid, ID].nunique()
        best_people = max(best_people, ppl)
        print(f"   {col:<10}  {100*valid.mean():5.1f}% of visits   {ppl:,} people")
    flag = "OK" if best_people >= 0.5*ids else "THIN - check"
    print(f"   -> reachable for ~{best_people:,} people   [{flag}]")

===== COGNITIVE DOMAIN COVERAGE (v3+v4) =====

Memory
   CRAFTVRS     79.2% of visits   30,102 people
   CRAFTURS     79.2% of visits   30,102 people
   CRAFTDVR     78.8% of visits   29,963 people
   CRAFTDRE     78.8% of visits   29,963 people
   UDSBENTD     75.6% of visits   30,536 people
   -> reachable for ~30,536 people   [OK]

Attention/WM
   DIGFORCT     80.3% of visits   30,558 people
   DIGFORSL     80.3% of visits   30,558 people
   DIGBACCT     80.1% of visits   30,480 people
   DIGBACLS     80.1% of visits   30,480 people
   TRAILA       76.2% of visits   30,824 people
   -> reachable for ~30,824 people   [OK]

Executive
   TRAILB       71.2% of visits   28,650 people
   UDSVERFC     84.2% of visits   31,178 people
   UDSVERLC     83.5% of visits   30,910 people
   UDSVERTN     83.4% of visits   30,879 people
   -> reachable for ~31,178 people   [OK]

Language
   MINTTOTS     72.0% of visits   29,839 people
   ANIMALS      86.0% of visits   31,811 people
   VEG          8

In [15]:
g = uds.groupby(ID)
def ever(col, val):
    return int((g[col].max() == val).sum()) if col in uds.columns else "col missing"

print("===== DIAGNOSIS / PATHOLOGY (per person) =====")
if "NACCUDSD" in uds.columns:
    lab = {1:"Normal", 2:"Impaired-not-MCI", 3:"MCI", 4:"Dementia", 8:"Behaviorally impaired"}
    worst = g["NACCUDSD"].max()
    print("Most severe status reached:")
    for k, n in worst.value_counts().sort_index().items():
        print(f"   {lab.get(k, k)}: {n:,}")
print(f"\nAlzheimer's (NACCALZD=1):  {ever('NACCALZD', 1)}")
print(f"LBD module (NACCLBDM=1):   {ever('NACCLBDM', 1)}   <-- Lewy-body sample")
print(f"FTLD module (NACCFTD=1):   {ever('NACCFTD', 1)}")
if "NACCETPR" in uds.columns:
    print("\nPrimary etiologic dx (NACCETPR), visit-level counts:")
    print(uds["NACCETPR"].value_counts(dropna=False).sort_index().head(20).to_string())

===== DIAGNOSIS / PATHOLOGY (per person) =====
Most severe status reached:
   Normal: 13,727
   Impaired-not-MCI: 1,737
   MCI: 7,916
   Dementia: 10,863
   Behaviorally impaired: 30

Alzheimer's (NACCALZD=1):  11397
LBD module (NACCLBDM=1):   811   <-- Lewy-body sample
FTLD module (NACCFTD=1):   2686

Primary etiologic dx (NACCETPR), visit-level counts:
NACCETPR
1     31284
2      2789
3        16
4       333
5       350
6       105
7      3536
8      2808
9        12
10       10
11        3
12       34
13      387
14       69
15       50
16       23
17        9
18     2615
19     1114
20      136


In [18]:
# most recent etiologic diagnosis per person
last = uds.sort_values("NACCVNUM").groupby(ID)["NACCETPR"].last()
lab = {1:"AD", 2:"LBD (incl. PDD/DLB)", 7:"FTLD-other", 8:"Vascular"}
print(last.map(lambda x: lab.get(x, x)).value_counts().head(10))

NACCETPR
88                     15234
AD                     11490
FTLD-other              1347
LBD (incl. PDD/DLB)     1084
30                      1043
Vascular                1014
18                       842
99                       496
19                       384
28                       215
Name: count, dtype: int64


In [20]:
# load the vascular-contribution column (wasn't in our original WANT set)
vcol = pd.read_csv(PATH, usecols=["NACCID", "FORMVER", "CVDIF"], low_memory=False)
v = vcol["FORMVER"].astype(str).str.strip()
vcol = vcol[v.str.startswith("3") | v.str.startswith("4")]
vcol["CVDIF"] = pd.to_numeric(vcol["CVDIF"], errors="coerce")

# per person: was vascular ever primary (1) or contributing (2)?
vasc_ever = vcol.groupby("NACCID")["CVDIF"].apply(lambda s: s.isin([1, 2]).any())

for code, name in [(1, "AD"), (2, "LBD")]:
    arm = last[last == code].index
    flags = vasc_ever.reindex(arm).fillna(False)
    mixed = int(flags.sum())
    print(f"{name}: {len(arm)} people | {mixed} with vascular involvement ({100*mixed/len(arm):.0f}%)")

AD: 11490 people | 1608 with vascular involvement (14%)
LBD: 1084 people | 82 with vascular involvement (8%)


In [22]:
print("===== COGNITIVE DOMAIN COVERAGE (v3+v4) =====")
print("(visit% = share of visits with a valid score; people = participants with >=1 valid score)")
for domain, cols in NEURO.items():
    print(f"\n{domain}")
    best_people = 0
    for col in cols:
        if col not in uds.columns:
            print(f"   {col:<10}  not in file")
            continue
        lo, hi = RANGES[col]
        valid = uds[col].between(lo, hi)
        ppl = uds.loc[valid, ID].nunique()
        best_people = max(best_people, ppl)
        print(f"   {col:<10}  {100*valid.mean():5.1f}% of visits   {ppl:,} people")
    flag = "OK" if best_people >= 0.5*ids else "THIN - check"
    print(f"   -> reachable for ~{best_people:,} people   [{flag}]")

===== COGNITIVE DOMAIN COVERAGE (v3+v4) =====
(visit% = share of visits with a valid score; people = participants with >=1 valid score)

Memory
   CRAFTVRS     79.2% of visits   30,102 people
   CRAFTURS     79.2% of visits   30,102 people
   CRAFTDVR     78.8% of visits   29,963 people
   CRAFTDRE     78.8% of visits   29,963 people
   UDSBENTD     75.6% of visits   30,536 people
   -> reachable for ~30,536 people   [OK]

Attention/WM
   DIGFORCT     80.3% of visits   30,558 people
   DIGFORSL     80.3% of visits   30,558 people
   DIGBACCT     80.1% of visits   30,480 people
   DIGBACLS     80.1% of visits   30,480 people
   TRAILA       76.2% of visits   30,824 people
   -> reachable for ~30,824 people   [OK]

Executive
   TRAILB       71.2% of visits   28,650 people
   UDSVERFC     84.2% of visits   31,178 people
   UDSVERLC     83.5% of visits   30,910 people
   UDSVERTN     83.4% of visits   30,879 people
   -> reachable for ~31,178 people   [OK]

Language
   MINTTOTS     72.0% o